# 02 — Grid Analytics: Gold Table Creation
This notebook reads from the Bronze Delta tables in the Lakehouse and creates
Gold aggregation tables for Power BI dashboards.

In [ ]:
from pyspark.sql import functions as F

# Read source tables from Lakehouse
df_sensors = spark.read.format("delta").table("grid_sensors")
df_events = spark.read.format("delta").table("power_events")
df_renewable = spark.read.format("delta").table("renewable_generation")

print(f"Sensors: {df_sensors.count()}, Events: {df_events.count()}, Renewable: {df_renewable.count()}")

In [ ]:
# Gold table: Hourly grid health by substation
gold_grid = df_sensors \
    .withColumn("date", F.to_date(F.col("timestamp"))) \
    .withColumn("hour", F.hour(F.col("timestamp"))) \
    .groupBy("date", "hour", "substation_id", "region") \
    .agg(
        F.round(F.avg("voltage_v"), 2).alias("avg_voltage"),
        F.round(F.avg("frequency_hz"), 3).alias("avg_frequency"),
        F.round(F.avg("load_mw"), 2).alias("avg_load"),
        F.round(F.avg("power_factor"), 3).alias("avg_power_factor"),
        F.round(F.avg("temperature_c"), 1).alias("avg_temperature"),
        F.count("*").alias("reading_count"),
        F.sum(F.when((F.col("voltage_v") < 220) | (F.col("voltage_v") > 240), 1).otherwise(0)).alias("voltage_anomalies"),
    ) \
    .withColumn("date", F.col("date").cast("string"))

gold_grid.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("gold_grid_health")
print(f"Gold grid health: {gold_grid.count()} rows")

In [ ]:
# Gold table: Daily outage/event summary by region
gold_outages = df_events \
    .withColumn("date", F.to_date(F.col("timestamp")).cast("string")) \
    .groupBy("date", "region") \
    .agg(
        F.count("*").alias("total_events"),
        F.sum(F.when(F.col("event_type") == "outage", 1).otherwise(0)).alias("outages"),
        F.sum(F.when(F.col("event_type") == "surge", 1).otherwise(0)).alias("surges"),
        F.sum(F.when(F.col("event_type") == "voltage_sag", 1).otherwise(0)).alias("sags"),
        F.sum(F.when(F.col("event_type") == "equipment_fault", 1).otherwise(0)).alias("faults"),
        F.sum("affected_customers").alias("total_affected"),
        F.round(F.avg("duration_sec") / 60.0, 1).alias("avg_duration_min"),
        F.sum(F.when(F.col("severity") == "critical", 1).otherwise(0)).alias("critical_events"),
    )

gold_outages.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("gold_outage_summary")
print(f"Gold outage summary: {gold_outages.count()} rows")

In [ ]:
# Gold table: Daily renewable generation summary by plant type
gold_renewable = df_renewable \
    .withColumn("date", F.to_date(F.col("timestamp")).cast("string")) \
    .groupBy("date", "plant_type") \
    .agg(
        F.round(F.sum("generation_mw"), 1).alias("total_generation_mw"),
        F.round(F.avg("capacity_factor"), 3).alias("avg_capacity_factor"),
        F.round(F.sum("capacity_mw"), 1).alias("total_capacity_mw"),
        F.count("*").alias("readings"),
    )

gold_renewable.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("gold_renewable_summary")
print(f"Gold renewable summary: {gold_renewable.count()} rows")
print("All Gold tables created successfully!")